### Parallel chains

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [13]:
# TASK 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system","You are a movie summerizer chatbot"),
        ("human","Write a summary of {input} movie in brief.")
    ]
)

In [14]:
# TASK 2 [llm]

llm = ChatGroq(model = "qwen/qwen3-32b" , temperature = 0)

In [15]:
# TASK 3 [parser]

str_parser = StrOutputParser()

In [16]:
# TASK 4 [custom runable]

from langchain_core.runnables import RunnableLambda

def dict_maker(text:str) ->dict:
    return {"text":text}

dictionary_maker_runnable = RunnableLambda(dict_maker)

### Parallel Chain 1

In [17]:
# Task 1 [prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a linked in post generator"),
    ("human","create a post for following text for linkedin : {text}")
])

# Task 2 [llm]
llm = ChatGroq(model = "qwen/qwen3-32b", temperature = 0)

# Task 3 [parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

In [18]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

### Parallel Chain 2

In [25]:
def insta_chain(text:dict):
    
    text = text["text"]
    
    # Task 1 [prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","you are a smart insta post generator"),
        ("human","create a post for the following text for instagram : {text}")
    ])
    
    # Task 2 [llm]
    llm = ChatGroq(model = "qwen/qwen3-32b", temperature = 0)
    
    # Task 3 [parser]
    str_parser = StrOutputParser()  
    
    chat_insta = insta_prompt | llm | str_parser
    result = chat_insta.invoke({"text":text})
    
    return result

insta_chain_runnable = RunnableLambda(insta_chain)


### Final Orchestration

In [ ]:
final_chain = (
                prompt_template |
                llm |
                str_parser |
                dictionary_maker_runnable |
                RunnableParallel(branches = 
                                 {
                                     "linkedin": chain_linkedin , 
                                     "instagram": insta_chain_runnable
                                 }
                                 )
                )

In [34]:
final_chain.invoke("Marco")

{'branches': {'linkedin': '<think>\nOkay, the user wants a LinkedIn post based on the movie summary I provided. Let me start by understanding the original summary. The movie is "Marco" from 2023, directed by Daniel Goldhaber. It\'s a coming-of-age story about a teenager dealing with identity, family issues, and self-discovery. The user probably wants a professional yet engaging LinkedIn post that highlights the film\'s themes and maybe relates it to personal or professional growth.\n\nFirst, I need to make sure the post is concise and fits LinkedIn\'s style. LinkedIn users often appreciate posts that are informative, thought-provoking, and sometimes include a call to action or a question to encourage engagement. Since the movie deals with themes like self-discovery and authenticity, I can tie those into broader life or career lessons.\n\nI should start with a catchy headline or a hook to grab attention. Maybe something like "Just watched *Marco* (2023)..." to make it personal. Then, me